# CaliRoute: Calibration-Aware Qubit Routing & Drift Forecasting

**CaliRoute** optimizes qubit layout and routing on superconducting quantum processors (IBM Quantum) using classical Graph Neural Networks (GNNs) and time-series calibration drift forecasting (LSTMs).

---

## 1. Load Real IBM Kingston 156-Qubit Hardware Telemetry

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

from src.collect.snapshot_calibration import get_latest_calibration_snapshot
from src.train.train_scorer import get_operational_physical_qubits

snapshot = get_latest_calibration_snapshot("../data/raw/calibration")
print(f"Backend: {snapshot['metadata']['backend_name']}")
print(f"Total Physical Qubits: {snapshot['metadata']['num_qubits']}")
print(f"Total Physical Coupling Edges: {len(snapshot['edges'])}")

operational = get_operational_physical_qubits(snapshot)
dead_qubits = [q for q in range(snapshot['metadata']['num_qubits']) if q not in set(operational)]
print(f"Operational Qubits: {len(operational)}")
print(f"Dead / Faulty Qubits Avoided: {dead_qubits}")

## 2. Load 30-Day Historical Data & Forecast Calibration Drift

In [ ]:
import torch
from src.models.drift_forecaster import CalibrationDriftForecaster

ckpt = torch.load("../results/drift_forecaster_best.pt", map_location="cpu", weights_only=False)
q_gain = ckpt['qubit_best_metrics']['mae_reduction_pct']
c_gain = ckpt['coupler_best_metrics']['mae_reduction_pct']

print(f"Qubit Drift Forecaster Gain over Persistence: {q_gain:+.2f}%")
print(f"Coupler Drift Forecaster Gain over Persistence: {c_gain:+.2f}%")

## 3. Compare CaliRoute vs. Qiskit SABRE Baseline

In [ ]:
import pandas as pd
df = pd.read_csv("../results/caliroute_vs_sabre.csv")
display_cols = ["circuit_id", "num_qubits", "sabre_fidelity", "caliroute_fidelity", "caliroute_faulty_gates"]
print(df[display_cols].to_string(index=False))

print(f"\nMean SABRE Fidelity: {df['sabre_fidelity'].mean():.4f}")
print(f"Mean CaliRoute Fidelity: {df['caliroute_fidelity'].mean():.4f}")